In [20]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import to_dense_adj, negative_sampling
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("✅ Setup Complete!", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup Complete! cuda


In [21]:
DATA_ROOT = '/content/drive/MyDrive/citeseer/citeseer'

transform = T.Compose([
    T.NormalizeFeatures(),
    T.RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True,
                      add_negative_train_samples=True, split_labels=True)
])

dataset = Planetoid(root=DATA_ROOT, name='CiteSeer', transform=transform)
train_data, val_data, test_data = dataset[0]

print(f"Nodes: {train_data.num_nodes}, Edges: {train_data.num_edges}")
print("✅ Data Loaded!")

Nodes: 3327, Edges: 7284
✅ Data Loaded!


In [22]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=32):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

teacher = GraphSAGE(train_data.x.shape[1], 64, 32).to(device)
optimizer_teacher = optim.Adam(teacher.parameters(), lr=0.01, weight_decay=5e-4)
print("✅ Teacher Ready")

✅ Teacher Ready


In [24]:
def train_teacher(epochs=800):
    teacher.train()
    for epoch in range(epochs):
        optimizer_teacher.zero_grad()
        z = teacher(train_data.x.to(device), train_data.edge_index.to(device))

        pos_edge = train_data.pos_edge_label_index.to(device)
        neg_edge = negative_sampling(train_data.edge_index,
                                     num_nodes=train_data.num_nodes,
                                     num_neg_samples=pos_edge.size(1)).to(device)

        src, dst = pos_edge
        pos_pred = (z[src] * z[dst]).sum(dim=1)
        src, dst = neg_edge
        neg_pred = (z[src] * z[dst]).sum(dim=1)

        loss = F.binary_cross_entropy_with_logits(
            torch.cat([pos_pred, neg_pred]),
            torch.cat([torch.ones(pos_pred.size(0), device=device),
                       torch.zeros(neg_pred.size(0), device=device)])
        )
        loss.backward()
        optimizer_teacher.step()

        if epoch % 100 == 0:
            print(f"Teacher Epoch {epoch} | Loss: {loss.item():.4f}")

train_teacher()
print("✅ Teacher Trained!")

Teacher Epoch 0 | Loss: 0.5318
Teacher Epoch 100 | Loss: 0.5275
Teacher Epoch 200 | Loss: 0.5356
Teacher Epoch 300 | Loss: 0.5322
Teacher Epoch 400 | Loss: 0.5280
Teacher Epoch 500 | Loss: 0.5300
Teacher Epoch 600 | Loss: 0.5321
Teacher Epoch 700 | Loss: 0.5407
✅ Teacher Trained!


In [42]:
def compute_high_order_adjs(data, max_order=2):
    adj = to_dense_adj(data.edge_index, max_num_nodes=data.num_nodes)[0].to(device)
    adj = adj + torch.eye(data.num_nodes, device=device)

    # Stronger normalization + clipping
    row_sum = adj.sum(dim=1, keepdim=True) + 1e-8
    adj = adj / row_sum
    adj = torch.clamp(adj, max=10.0)

    high_order = {1: adj}
    for k in range(2, max_order+1):
        high_order[k] = torch.mm(high_order[k-1], adj)
        row_sum = high_order[k].sum(dim=1, keepdim=True) + 1e-8
        high_order[k] = high_order[k] / row_sum
        high_order[k] = torch.clamp(high_order[k], max=5.0)

    print(f"✅ High-order adjs computed up to A^{max_order} (Strong Normalization)")
    return high_order

high_order = compute_high_order_adjs(train_data)

class SEN(nn.Module):
    def __init__(self, hidden_dim=16, out_dim=32):   # Further reduced
        super().__init__()
        self.f_edge = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.LeakyReLU()
        )
        self.f_node = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.LeakyReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, adj_l):
        x = adj_l.unsqueeze(-1)
        edge_feat = self.f_edge(x)
        node_feat = edge_feat.sum(dim=1)
        return self.f_node(node_feat)

print("✅ SEN Ready (Stabilized)")

✅ High-order adjs computed up to A^2 (Strong Normalization)
✅ SEN Ready (Stabilized)


In [45]:
class HSAD_Student(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LeakyReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
        self.sen = SEN(hidden_dim=16, out_dim=out_dim)
        self.beta = nn.Parameter(torch.tensor(0.7))
        # Removed SA_Attn for stability (can add later)

    def forward(self, x, high_order_adjs):
        z = self.mlp(x)
        sal = self.sen(high_order_adjs[2])
        return z, sal, z   # Return z as z_attn for now

student = HSAD_Student(train_data.x.shape[1]).to(device)
print("✅ HSAD Student Ready (Stabilized Version)")

✅ HSAD Student Ready (Stabilized Version)


In [46]:
optimizer = optim.Adam(student.parameters(), lr=0.002, weight_decay=5e-4)

def train_epoch():
    student.train()
    optimizer.zero_grad()

    train_x = train_data.x.to(device)
    edge_index = train_data.edge_index.to(device)

    z, sal, _ = student(train_x, high_order)

    pos = train_data.pos_edge_label_index.to(device)
    neg = train_data.neg_edge_label_index.to(device)
    edge_idx = torch.cat([pos, neg], dim=1)
    src, dst = edge_idx

    edge_pred = (z[src] * z[dst]).sum(dim=1) * student.beta + \
                ((1 - student.beta) * (sal[src] * sal[dst]).sum(dim=1))

    labels = torch.cat([torch.ones(pos.size(1)), torch.zeros(neg.size(1))]).to(device)
    loss_edge = F.binary_cross_entropy_with_logits(edge_pred, labels)

    # Teacher Distillation
    with torch.no_grad():
        teacher_z = teacher(train_x, edge_index)

    loss_node = F.mse_loss(z, teacher_z.detach())
    teacher_edge_pred = (teacher_z[src] * teacher_z[dst]).sum(dim=1)
    teacher_prob = torch.sigmoid(teacher_edge_pred).detach()
    loss_kd = F.binary_cross_entropy_with_logits(edge_pred, teacher_prob)

    loss_total = loss_edge + 0.3 * loss_node + 0.5 * loss_kd

    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
    optimizer.step()
    return loss_total.item(), loss_edge.item()

print("🚀 Starting Stabilized HSAD Training...\n")
for epoch in range(500):
    loss_total, loss_edge = train_epoch()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Total Loss: {loss_total:.4f} | Edge Loss: {loss_edge:.4f}")

🚀 Starting Stabilized HSAD Training...

Epoch   0 | Total Loss: 1.2970 | Edge Loss: 0.9220
Epoch  50 | Total Loss: 1.0429 | Edge Loss: 0.6974
Epoch 100 | Total Loss: 1.0429 | Edge Loss: 0.6973
Epoch 150 | Total Loss: 1.0429 | Edge Loss: 0.6973
Epoch 200 | Total Loss: 0.9993 | Edge Loss: 0.6627
Epoch 250 | Total Loss: 0.8614 | Edge Loss: 0.5223
Epoch 300 | Total Loss: 0.8378 | Edge Loss: 0.4984
Epoch 350 | Total Loss: 0.8280 | Edge Loss: 0.4886
Epoch 400 | Total Loss: 0.8220 | Edge Loss: 0.4827
Epoch 450 | Total Loss: 0.8043 | Edge Loss: 0.4548


In [47]:
@torch.no_grad()
def evaluate(data_split, k=50):
    student.eval()
    data_x = data_split.x.to(device)
    z, sal, _ = student(data_x, high_order)

    pos = data_split.pos_edge_label_index.to(device)
    neg = data_split.neg_edge_label_index.to(device)
    edge_idx = torch.cat([pos, neg], dim=1)
    src, dst = edge_idx

    pred = (z[src] * z[dst]).sum(dim=1) * student.beta + \
           ((1 - student.beta) * (sal[src] * sal[dst]).sum(dim=1))

    prob = torch.sigmoid(pred).cpu().numpy()
    labels = torch.cat([torch.ones(pos.size(1)), torch.zeros(neg.size(1))]).cpu().numpy()

    auc = roc_auc_score(labels, prob)
    ap = average_precision_score(labels, prob)

    pred_tensor = torch.tensor(prob)
    label_tensor = torch.tensor(labels)
    _, indices = torch.sort(pred_tensor, descending=True)
    hits_k = label_tensor[indices[:k]].float().mean().item()

    return auc, ap, hits_k

test_auc, test_ap, test_hits50 = evaluate(test_data, k=50)
val_auc, val_ap, val_hits50 = evaluate(val_data, k=50)

print("="*70)
print("🎉 FINAL RESULTS - CITESEER")
print(f"Test  →  AUC: {test_auc:.4f} | AP: {test_ap:.4f} | Hits@50: {test_hits50:.4f}")
print(f"Val   →  AUC: {val_auc:.4f} | AP: {val_ap:.4f} | Hits@50: {val_hits50:.4f}")
print("="*70)

🎉 FINAL RESULTS - CITESEER
Test  →  AUC: 0.7680 | AP: 0.7667 | Hits@50: 0.9400
Val   →  AUC: 0.7591 | AP: 0.7607 | Hits@50: 0.9600
